# 02 – Data Preprocessing and Pipeline Preparation

**ENGE707 Assessment 3 – Facial Feature Classification Challenge (Kaggle)**

**Purpose.** Build and test, step by step, the preprocessing pipeline described in report Task 2 / Section 2: label
encoding, the stratified train/validation split, resizing and normalisation, `tf.data` loading, data augmentation, the
leakage check and the identical test-set pipeline. The final, reusable version of this code lives in `../src/pipeline.py`,
which every later notebook imports so that the training, validation and test paths cannot drift apart.

**Inputs.** `../data/train_labels.csv`, `../data/train/train/*.jpg`, `../data/sample_submission.csv`,
`../data/test/test/*.jpg`.

**Outputs.** Printed checks (shapes, pixel ranges, class proportions, zero filename overlap) and one augmentation preview.
No files are written.

**How to run.** Start Jupyter from the `notebooks/` folder and run the cells top to bottom.

## Step 1: Encode the class labels

Turns the eight text class names into the integers 0–7. The names are sorted alphabetically first, so the mapping is the same every time the notebook is run (CatEye = 0 … DogNose = 7). Report Section 2.1.

In [ ]:
import pandas as pd  # pandas: tables

labels = pd.read_csv("../data/train_labels.csv")  # load the label file (image_id, class)

# Sort alphabetically for a consistent, reproducible mapping
class_names = sorted(labels['class'].unique())  # the eight class names in alphabetical order
class_to_idx = {name: idx for idx, name in enumerate(class_names)}  # dictionary: class name -> integer (0..7)
idx_to_class = {idx: name for name, idx in class_to_idx.items()}  # reverse dictionary: integer -> class name

labels['label'] = labels['class'].map(class_to_idx)  # new column 'label' with the integer code of each row's class

print(class_to_idx)  # show the mapping so it can be checked
labels.head()  # show the first 5 rows, now including the integer label

## Step 2: Build the image filenames

The label file stores `image_id` as a plain integer (1, 2, 3 …) but the image files on disk are named with five digits (`00001.jpg`). This cell builds a matching `filename` column so every row points to its image file.

In [ ]:
# format each integer id as a 5-digit zero-padded number and add '.jpg' (e.g. 1 -> '00001.jpg')
labels['filename'] = labels['image_id'].apply(lambda x: f"{x:05d}.jpg")
labels.head()  # show the first 5 rows to confirm the new column

## Step 3: Stratified 80/20 train/validation split

Splits the 63,660 labelled images into 50,928 training and 12,732 validation images. `stratify` keeps the class proportions the same in both subsets (important because the classes are imbalanced) and `random_state=42` makes the split repeatable. The printed proportions confirm the two subsets match. Report Section 2.1.

In [8]:
from sklearn.model_selection import train_test_split  # scikit-learn helper that splits data at random

train_df, val_df = train_test_split(  # returns the training rows and the validation rows
    labels,  # the table to split
    test_size=0.2,  # 20% of the rows go to the validation set
    stratify=labels['label'],  # keep the same class proportions in both subsets
    random_state=42  # fixed seed so the split is identical every run
)

print(f"Train: {len(train_df)} images")  # number of training images
print(f"Val:   {len(val_df)} images")  # number of validation images

# Confirm class proportions match between the two sets
print("\nTrain class proportions:")  # heading for the next output
print(train_df['class'].value_counts(normalize=True).sort_index())  # share of each class in the training set
print("\nVal class proportions:")  # heading for the next output
print(val_df['class'].value_counts(normalize=True).sort_index())  # share of each class in the validation set

Train: 50928 images
Val:   12732 images

Train class proportions:
class
CatEye      0.218092
CatFace     0.109056
CatMouth    0.109056
CatNose     0.109036
DogEye      0.181904
DogFace     0.090952
DogMouth    0.090952
DogNose     0.090952
Name: proportion, dtype: float64

Val class proportions:
class
CatEye      0.218112
CatFace     0.109017
CatMouth    0.109017
CatNose     0.109095
DogEye      0.181904
DogFace     0.090952
DogMouth    0.090952
DogNose     0.090952
Name: proportion, dtype: float64


## Step 4: Load, resize and normalise one image

Defines the function that turns an image file into model input: read the file, decode the JPEG, resize to 224x224 and scale the pixel values from 0–255 to 0–1. It is tested on a single image. Report Section 2.2.

In [9]:
import tensorflow as tf  # TensorFlow: image decoding and the tf.data input pipeline

IMG_SIZE = 224  # side length (pixels) expected by the pretrained networks
IMG_DIR = "../data/train/train"  # folder with the training images

def load_and_preprocess(filename, label):  # takes a filename and its label, returns (image tensor, label)
    path = IMG_DIR + "/" + filename  # full path of the image file
    image = tf.io.read_file(path)  # read the raw bytes of the file
    image = tf.image.decode_jpeg(image, channels=3)  # decode the JPEG into a height x width x 3 (RGB) array
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])  # resize to 224 x 224 (small images are upscaled)
    image = image / 255.0  # normalize to [0,1]
    return image, label  # hand back the processed image with its unchanged label

# Quick test on a single file
test_img, test_label = load_and_preprocess(train_df.iloc[0]['filename'], train_df.iloc[0]['label'])  # process the first training row
print(test_img.shape, test_img.dtype)  # expect (224, 224, 3) and float32
print("min/max pixel value:", float(tf.reduce_min(test_img)), float(tf.reduce_max(test_img)))  # expect values within 0..1

(224, 224, 3) <dtype: 'float32'>
min/max pixel value: 0.0 1.0


## Step 5: Build the batched, shuffled, prefetched datasets

Wraps the loading function in a `tf.data` pipeline so images are streamed from disk in batches of 32 instead of being held in memory: parallel decoding, shuffling for the training set only, batching and prefetching. One batch is pulled to check the shapes. Report Section 2.2.

In [10]:
BATCH_SIZE = 32  # number of images processed per optimisation step

def make_dataset(df, shuffle=False):  # build a tf.data pipeline from a DataFrame of filenames and labels
    filenames = df['filename'].values  # array of filenames
    labels_arr = df['label'].values  # array of integer labels (same order)
    ds = tf.data.Dataset.from_tensor_slices((filenames, labels_arr))  # dataset of (filename, label) pairs
    if shuffle:  # only the training set is shuffled
        ds = ds.shuffle(buffer_size=len(df), seed=42)  # random order, buffer as large as the whole set, fixed seed
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)  # load/resize/normalise images in parallel
    ds = ds.batch(BATCH_SIZE)  # group 32 images into one batch
    ds = ds.prefetch(tf.data.AUTOTUNE)  # prepare the next batch while the GPU works on the current one
    return ds  # return the finished pipeline

train_ds = make_dataset(train_df, shuffle=True)  # training pipeline (shuffled)
val_ds = make_dataset(val_df, shuffle=False)  # validation pipeline (fixed order)

# Quick check: pull one batch and confirm shapes
for images, batch_labels in train_ds.take(1):  # take just the first batch
    print("Batch images shape:", images.shape)  # expect (32, 224, 224, 3)
    print("Batch labels shape:", batch_labels.shape)  # expect (32,)
    print("Sample labels:", batch_labels.numpy()[:10])  # first 10 integer labels of the batch

Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Sample labels: [7 6 0 2 3 5 6 6 1 7]


## Step 6: Leakage check: filename overlap between train and validation

Intersects the set of training filenames with the set of validation filenames. The overlap must be 0 so that no image is in both subsets. This is a file-level check only: it cannot detect different crops of the same photograph (discussed in report Sections 2.3 and 6.4).

In [11]:
overlap = set(train_df['filename']) & set(val_df['filename'])  # filenames that appear in BOTH subsets
print(f"Overlapping filenames between train and val: {len(overlap)}")  # must be 0

Overlapping filenames between train and val: 0


## Step 7: Data augmentation (training set only)

Defines the augmentation layers (horizontal flip, rotation about ±18°, small brightness and contrast changes), applies them to the training pipeline and plots five original images above their augmented versions to check the changes are subtle and the feature is still visible. Report Section 2.3.

In [12]:
import matplotlib.pyplot as plt  # ADDED: this notebook plots below but never imported plt, so a fresh kernel failed

data_augmentation = tf.keras.Sequential([  # a chain of random image transformations
    tf.keras.layers.RandomFlip("horizontal"),  # randomly mirror the image left-right
    tf.keras.layers.RandomRotation(0.05),  # random rotation of up to 5% of a full turn (about +/-18 degrees)
    tf.keras.layers.RandomBrightness(0.1, value_range=(0.0, 1.0)),  # brightness change up to 10%; tell it pixels are in 0..1
    tf.keras.layers.RandomContrast(0.1),  # random contrast change of up to 10%
])

def augment(image, label):  # apply the augmentation to a batch of images
    image = data_augmentation(image, training=True)  # training=True switches the random layers on
    image = tf.clip_by_value(image, 0.0, 1.0)  # safety net against any small overshoot
    return image, label  # return the augmented images with unchanged labels

train_ds_aug = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)  # augmented version of the training pipeline

for images, labels_batch in train_ds.take(1):  # take one batch of un-augmented images
    aug_images, _ = augment(images, labels_batch)  # augment that same batch (labels are not needed here)
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))  # 2 rows (original / augmented) x 5 columns
    for i in range(5):  # show the first five images of the batch
        axes[0, i].imshow(images[i])  # top row: original image
        axes[0, i].set_title(f"Original: {idx_to_class[labels_batch[i].numpy()]}")  # title with its class name
        axes[0, i].axis('off')  # hide axes
        axes[1, i].imshow(aug_images[i])  # bottom row: the augmented version
        axes[1, i].set_title("Augmented")  # title
        axes[1, i].axis('off')  # hide axes
    plt.tight_layout()  # tidy the spacing
    plt.show()  # display the comparison

NameError: name 'plt' is not defined

## Step 8: Test filenames from the sample submission

Loads `sample_submission.csv`, which lists every test `image_id` in the order Kaggle expects, and builds the same zero-padded filenames used for the training images. Report Section 2.2.

In [13]:
import os  # os is imported here but not actually used in this notebook

TEST_DIR = "../data/test/test"  # folder with the unlabelled test images
sample_sub = pd.read_csv("../data/sample_submission.csv")  # Kaggle's template: image_id and a placeholder class

# Build the same zero-padded filename format used for train
sample_sub['filename'] = sample_sub['image_id'].apply(lambda x: f"{x:05d}.jpg")  # e.g. 1 -> '00001.jpg'

print(sample_sub.shape)  # expect (29443, 3): one row per test image
sample_sub.head()  # show the first 5 rows

(29443, 3)


,image_id,class,filename
0,1,DogEye,00001.jpg
1,2,CatNose,00002.jpg
2,3,CatNose,00003.jpg
3,4,DogFace,00004.jpg
4,5,DogMouth,00005.jpg


## Step 9: Test-set pipeline (no labels, no shuffling, no augmentation)

Builds the test dataset with exactly the same resizing and scaling as training/validation, but without shuffling (so predictions stay in the order of `sample_submission.csv`) and without augmentation (so the test set measures unaltered performance). Report Section 2.2.

In [14]:
def load_test_image(filename):  # like load_and_preprocess, but test images have no label
    path = TEST_DIR + "/" + filename  # full path of the test image
    image = tf.io.read_file(path)  # read the raw file bytes
    image = tf.image.decode_jpeg(image, channels=3)  # decode to an RGB array
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])  # resize to 224 x 224, identical to training
    image = image / 255.0  # scale pixel values to [0,1], identical to training
    return image  # return only the image

test_filenames = sample_sub['filename'].values  # filenames in the exact Kaggle submission order

test_ds = tf.data.Dataset.from_tensor_slices(test_filenames)  # dataset of filenames (no shuffling, so order is kept)
test_ds = test_ds.map(load_test_image, num_parallel_calls=tf.data.AUTOTUNE)  # load and preprocess in parallel
test_ds = test_ds.batch(BATCH_SIZE)  # batches of 32
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)  # overlap data preparation with prediction

# Quick check
for batch in test_ds.take(1):  # take the first batch
    print("Test batch shape:", batch.shape)  # expect (32, 224, 224, 3)

Test batch shape: (32, 224, 224, 3)
